# DICOM

In [ ]:
import os
import numpy as np
import pydicom
import plotly.express as px

path_ = r"c:\Users\bguet\Desktop\data\ARTIX\DICOM_ARTIX_data\001\CT0\DOE^JOHN_ANON61841_CT_2017-06-21_100509_ORL.(sauf.sinus)_ORL.2MM_n198__00000"
read_dcm = lambda i : pydicom.dcmread(os.path.join(path_, i)).pixel_array
ct = list(map(read_dcm, sorted(os.listdir(path_))))
ct = np.stack(ct, axis=0)
print(ct.shape)

fig = px.imshow(
    ct, 
    animation_frame=0,
    binary_string=True, 
    labels=dict(animation_frame="slice"))
fig.show()

In [ ]:
# Manufacturer (0008, 0070)
# X-Ray Tube Current (0018,1151)
# Exposure (0018,1152)
# Exposure Time (0018,1150)
# Exposure = (X-Ray Tube Current) * (Exposure Time)

import os, glob, random
import pydicom

def print_recursiverly(path, data):
    try:
        is_dcm = all([pydicom.misc.is_dicom(j) for j in glob.glob(os.path.join(path, "*"))])
    except PermissionError:
        is_dcm = False
    
    if is_dcm:
        if len(glob.glob(os.path.join(path, "*"))) > 0:
            dcm = pydicom.dcmread(glob.glob(os.path.join(path, "*"))[0])
            if dcm.get((0x0008, 0x0060)).value == "CT":
                manufacturer = dcm.get((0x0008, 0x0070)).value
                xray_curr = dcm.get((0x0018, 0x1151)).value
                try:
                    exposure_t = dcm.get((0x0018, 0x1150)).value
                except (AttributeError, TypeError):
                    exposure_t = None
                try:
                    exposure = dcm.get((0x0018, 0x1152)).value
                except AttributeError:
                    exposure = None
                
                print(path)
                print(manufacturer)
                print(xray_curr)
                print(exposure_t)
                print(exposure)
                print()
                data.append({"manufacturer": manufacturer, "xray_curr": xray_curr, "exposure_t": exposure_t, 
                             "exposure": exposure, "path": path})
    else:
        for j in glob.glob(os.path.join(path, "*")):
            print_recursiverly(j, data)

data = []
N = 20

patients = glob.glob(os.path.join(r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data", "*"))
for patient in random.sample(patients, N):
    print(patient)
    for i in glob.glob(os.path.join(patient, "*")):
        print_recursiverly(i, data)

patients = glob.glob(os.path.join(r"C:\Users\bilel.guetarni\Desktop\data\TCIA\HNSCC-3DCT-RT\manifest-1549495779734\HNSCC-3DCT-RT", "*"))
for patient in random.sample(patients, N):
    print(patient)
    for i in glob.glob(os.path.join(patient, "*")):
        print_recursiverly(i, data)

print(len(data))

In [ ]:
df = pandas.DataFrame(data)
print(df)

idx = df.loc[pandas.isna(df["exposure_t"])].index
df.loc[idx, "exposure_t"] = (1000 * df.loc[idx, "exposure"]) / df.loc[idx, "xray_curr"]

idx = df.loc[pandas.isna(df["exposure"])].index
df.loc[idx, "exposure"] = df.loc[idx, "xray_curr"] * (df.loc[idx, "exposure_t"] / 1000)

print(df)

In [ ]:
for _, row in df.iterrows():
    print(row["path"])
    print(row)

In [54]:
for _, row in df[df["manufacturer"] == "GE MEDICAL SYSTEMS"].iterrows():
    print(row["path"])

C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data\016\extraPETCT\DOE^JANE_ANON47040_CT_2014-06-26_150116_TEP.ORL_CT.Standard.3.75_n287__00000
C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data\016\extraPETCT\DOE^JANE_ANON47040_CT_2014-07-16_133950_TEP.ORL_CT.Standard.3.75_n287__00000


In [ ]:
import plotly.express as px

fig = px.scatter(df, x="xray_curr", y="exposure_t", color="manufacturer")
fig.show()

fig = px.scatter(df, x="xray_curr", y="exposure", color="manufacturer")
fig.show()

In [ ]:
import plotly.express as px

for i in df["manufacturer"].unique():
    print(i)
    fig = px.histogram(df[df["manufacturer"] == i], x="exposure_t", nbins=10)
    fig.update_xaxes(range=[-0, 1200])
    fig.show()

In [ ]:
import pandas
import plotly.express as px

for type in df["type"].unique():
    dff = df[df["type"] == type]
    dff = [{"type": k, "value": len(dff[dff["value"] == k])} for k in dff["value"].unique()]
    dff = pandas.DataFrame(dff)
    print(dff)

    fig = px.pie(dff, values='value', names='type')
    fig.show()

In [ ]:
for i in df["value"].unique():
    print(i)
    dff = df[df["value"] == i]
    for i, row in dff.iterrows():
        print(row["path"])

In [ ]:
from datetime import datetime

datetime_object = datetime.strptime(f[0x0008, 0x0012].value, '%Y%m%d')
print(datetime_object.day)

In [ ]:
import glob, os

for i in glob.glob(os.path.join(r"c:\Users\bguet\Desktop\data\ARTIX\DICOM_ARTIX_data\004", "CT*")):
    print(i)
    for j in glob.glob(os.path.join(i, "*")):
        print(j)
        for k in glob.glob(os.path.join(j, "*"))[:1]:
            if pydicom.misc.is_dicom(k):
                dcm = pydicom.dcmread(k)
                print(datetime.strptime(dcm[0x0008, 0x0012].value, '%Y%m%d'))

In [ ]:
global XCURRENT
XCURRENT = []

def print_recursiverly(path):
    try:
        is_dcm = all([pydicom.misc.is_dicom(j) for j in glob.glob(os.path.join(path, "*"))])
    except PermissionError:
        is_dcm = False
    
    if is_dcm:
        if len(glob.glob(os.path.join(path, "*"))) > 0:
            dcm = pydicom.dcmread(glob.glob(os.path.join(path, "*"))[0])
            if dcm.get((0x0008, 0x0060)).value == "CT":
                type = "CBCT" if "CBCT" in os.path.split(path)[1] else "CT"
                value = dcm.get((0x0018, 0x1151)).value
                print(value)
                XCURRENT.append({"type": type, "value": value})
    else:
        for j in glob.glob(os.path.join(path, "*")):
            print_recursiverly(j)

for patient in glob.glob(os.path.join(r"c:\Users\bguet\Desktop\data\ARTIX\DICOM_ARTIX_data", "*")):
    print(patient)
    for i in glob.glob(os.path.join(patient, "*")):
        print_recursiverly(i)

In [ ]:
import pandas
import plotly.express as px
import plotly.figure_factory as ff

df = pandas.DataFrame(XCURRENT)

group_labels = ['CT', 'CBCT']
hist_data = [df[df["type"] == i]["value"] for i in group_labels]
fig = ff.create_distplot(hist_data, group_labels)
fig.show()

The tag (0020,0052) Frame of Reference UID is available in three different ways:
    (0020,0052) Frame of Reference UID
    (3006,0010) -> item -> (0020,0052) Frame of Reference UID
    (3006,0020) -> item -> (3006,0024) Referenced Frame of Reference UID

In [ ]:
import cv2
from dicom_class import CT, RTSTRUCT
import plotly.express as px

import numpy as np
from dicom_utils import fill_vol_ctrs

rtstruct = RTSTRUCT(r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data\002\CT5\DOE^JOHN_ANON74898_RTst_2017-07-17_133250_ORL.(sauf.sinus)_CT.5.DOSI.0_n1__00000\2.16.840.1.114362.1.11956109.23961857828.605402267.473.1571.dcm")

ct = CT(
    path=r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data\002\CT5\DOE^JOHN_ANON74898_CT_2017-07-17_133250_ORL.(sauf.sinus)_ORL.2MM.ARTIX_n200__00000",
    rtstruct=rtstruct
)

slide = 138
contour_name = "Encephale"

ctrs = ct.rtstruct.get_contours(contour_name)
mask = fill_vol_ctrs(ct.nii.shape, ctrs)
# print(mask.shape)

ct_img = ct.get_voxel_array()

src1 = np.moveaxis(ct_img, -1, 0)[slide]
src2 = np.moveaxis(mask.astype(ct_img.dtype), -1, 0)[slide]

src1 = ((src1 - np.min(src1)) / np.max(src1)) * 255
src2 = ((src2 - np.min(src2)) / np.max(src2)) * 255

img = cv2.addWeighted(src1, 0.5, src2, 0.5, 0.0)

fig = px.imshow(img)
fig.show()

# RTDOSE

In [71]:
from dicom_class import CT, RTDOSE
from dicompylercore import dicomparser, dvh, dvhcalc

ct = r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data\002\CT5\DOE^JOHN_ANON74898_CT_2017-07-17_133250_ORL.(sauf.sinus)_ORL.2MM.ARTIX_n200__00000"
dose = r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data\002\CT5\DOE^JOHN_ANON74898_RTDOSE_2017-07-17_133250_ORL.(sauf.sinus)_CT.5.DOSI.0_n1__00000\2.16.840.1.114362.1.11956109.23961857828.605402274.577.1575.dcm"
st = r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data\002\CT5\DOE^JOHN_ANON74898_RTst_2017-07-17_133250_ORL.(sauf.sinus)_CT.5.DOSI.0_n1__00000\2.16.840.1.114362.1.11956109.23961857828.605402267.473.1571.dcm"

In [72]:
# i.e. Get a dict of structure information
dp = dicomparser.DicomParser(st)
structures = dp.GetStructures()
structures

{1: {'id': 1,
  'name': 'ISO',
  'type': 'ISOCENTER',
  'color': array([255,   0,   0]),
  'empty': False},
 2: {'id': 2,
  'name': 'coordonnees table',
  'type': 'MARKER',
  'color': array([  0, 255,   0]),
  'empty': False},
 3: {'id': 3,
  'name': 'ATM_controlaterale',
  'type': 'ORGAN',
  'color': array([192, 255,   0]),
  'empty': False},
 4: {'id': 4,
  'name': 'ATM_homolaterale',
  'type': 'ORGAN',
  'color': array([  0, 128, 255]),
  'empty': False},
 5: {'id': 5,
  'name': 'Bouche_oesophagienne',
  'type': 'ORGAN',
  'color': array([200, 180, 255]),
  'empty': False},
 6: {'id': 6,
  'name': 'Canal_medullaire',
  'type': 'ORGAN',
  'color': array([165,  80,  65]),
  'empty': False},
 7: {'id': 7,
  'name': 'Encephale',
  'type': 'ORGAN',
  'color': array([255,   0, 255]),
  'empty': False},
 8: {'id': 8,
  'name': 'Constricteur_inf',
  'type': 'ORGAN',
  'color': array([  0, 160, 160]),
  'empty': False},
 9: {'id': 9,
  'name': 'Larynx',
  'type': 'ORGAN',
  'color': array([2

In [75]:
# Calculate a DVH from DICOM RT data
calcdvh = dvhcalc.get_dvh(st, dose, 17)
# calcdvh.relative_volume.plot()
print(calcdvh.mean)

21.37035337552729


In [78]:
dose1 = r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data\002\CT5\DOE^JOHN_ANON74898_RTDOSE_2017-07-17_133250_ORL.(sauf.sinus)_CT.5.DOSI.0_n1__00000\2.16.840.1.114362.1.11956109.23961857828.605402274.577.1575.dcm"
dose2 = r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data\002\CT5\DOE^JOHN_ANON74898_RTDOSE_2017-07-17_133250_ORL.(sauf.sinus)_CT.5.DOSI.5_n1__00000\2.16.840.1.114362.1.11956109.23961857828.605402276.1060.1577.dcm"
st = r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data\002\CT5\DOE^JOHN_ANON74898_RTst_2017-07-17_133250_ORL.(sauf.sinus)_CT.5.DOSI.0_n1__00000\2.16.840.1.114362.1.11956109.23961857828.605402267.473.1571.dcm"

dp = dicomparser.DicomParser(st)
for _, str in dp.GetStructures().items():
    print(str["name"])
    calcdvh = dvhcalc.get_dvh(st, dose1, str["id"])
    print(calcdvh.mean)
    calcdvh = dvhcalc.get_dvh(st, dose2, str["id"])
    print(calcdvh.mean)

ISO
0
0
coordonnees table
0
0
ATM_controlaterale
5.098058823529412
5.06252688172043
ATM_homolaterale
13.260445544554454
19.485105263157894
Bouche_oesophagienne
54.12264227642277
50.323606557377055
Canal_medullaire
23.48215909090895


Dose plane not found for -111.00. Using -108.601 to calculate contour volume.
Dose plane not found for -113.00. Using -108.601 to calculate contour volume.
Dose plane not found for -115.00. Using -108.601 to calculate contour volume.
Dose plane not found for -117.00. Using -108.601 to calculate contour volume.
Dose plane not found for -119.00. Using -108.601 to calculate contour volume.
Dose plane not found for -121.00. Using -108.601 to calculate contour volume.
Dose plane not found for -123.00. Using -108.601 to calculate contour volume.
Dose plane not found for -125.00. Using -108.601 to calculate contour volume.
Dose plane not found for -127.00. Using -108.601 to calculate contour volume.
Dose plane not found for -129.00. Using -108.601 to calculate contour volume.
Dose plane not found for -131.00. Using -108.601 to calculate contour volume.
Dose plane not found for -133.00. Using -108.601 to calculate contour volume.
Dose plane not found for -135.00. Using -108.601 to calculate co

24.48419949174116
Encephale


Dose plane not found for 101.00. Using -184.865 to calculate contour volume.
Dose plane not found for 103.00. Using -184.865 to calculate contour volume.
Dose plane not found for 105.00. Using -184.865 to calculate contour volume.
Dose plane not found for 107.00. Using -184.865 to calculate contour volume.
Dose plane not found for 109.00. Using -184.865 to calculate contour volume.
Dose plane not found for 111.00. Using -184.865 to calculate contour volume.
Dose plane not found for 113.00. Using -184.865 to calculate contour volume.
Dose plane not found for 115.00. Using -184.865 to calculate contour volume.
Dose plane not found for 117.00. Using -184.865 to calculate contour volume.
Dose plane not found for 119.00. Using -184.865 to calculate contour volume.
Dose plane not found for 121.00. Using -184.865 to calculate contour volume.
Dose plane not found for 123.00. Using -184.865 to calculate contour volume.
Dose plane not found for 125.00. Using -184.865 to calculate contour volume.

17.89980351602884


...

Dose plane not found for -111.00. Using -108.601 to calculate contour volume.
Dose plane not found for -113.00. Using -108.601 to calculate contour volume.
Dose plane not found for -115.00. Using -108.601 to calculate contour volume.
Dose plane not found for -117.00. Using -108.601 to calculate contour volume.
Dose plane not found for -119.00. Using -108.601 to calculate contour volume.
Dose plane not found for -121.00. Using -108.601 to calculate contour volume.
Dose plane not found for -123.00. Using -108.601 to calculate contour volume.
Dose plane not found for -125.00. Using -108.601 to calculate contour volume.
Dose plane not found for -127.00. Using -108.601 to calculate contour volume.
Dose plane not found for -129.00. Using -108.601 to calculate contour volume.
Dose plane not found for -131.00. Using -108.601 to calculate contour volume.
Dose plane not found for -133.00. Using -108.601 to calculate contour volume.
Dose plane not found for -135.00. Using -108.601 to calculate co

0
tableE-ext


Dose plane not found for 101.00. Using -184.865 to calculate contour volume.
Dose plane not found for 103.00. Using -184.865 to calculate contour volume.
Dose plane not found for 105.00. Using -184.865 to calculate contour volume.
Dose plane not found for 107.00. Using -184.865 to calculate contour volume.
Dose plane not found for 109.00. Using -184.865 to calculate contour volume.
Dose plane not found for 111.00. Using -184.865 to calculate contour volume.
Dose plane not found for 113.00. Using -184.865 to calculate contour volume.
Dose plane not found for 115.00. Using -184.865 to calculate contour volume.
Dose plane not found for 117.00. Using -184.865 to calculate contour volume.
Dose plane not found for 119.00. Using -184.865 to calculate contour volume.
Dose plane not found for 121.00. Using -184.865 to calculate contour volume.
Dose plane not found for 123.00. Using -184.865 to calculate contour volume.
Dose plane not found for 125.00. Using -184.865 to calculate contour volume.

2.518164204407853


KeyboardInterrupt: 

# clinical

In [ ]:
import pandas
import artix

df = pandas.read_csv(r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\toxicity_data\20241021_MDA_LTSI.csv", sep=";")
df = df[df["USUBJID"] == df["USUBJID"].unique()[1]]
print(df["USUBJID"].unique())
print(artix.mda_parsing(df))

In [ ]:
import pandas
import artix

df = pandas.read_csv(r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\toxicity_data\20241021_SALIVATION_DATA_LTSI.csv", sep=";")
df = df[df["USUBJID"] == df["USUBJID"].unique()[1]]
print(df["USUBJID"].unique())
print(artix.salivation_parsing(df))

In [ ]:
import pandas
import artix

df = pandas.read_csv(r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\toxicity_data\20241021_DOSIMETRY_LTSI.csv", sep=";")
df = df[df["USUBJID"] == df["USUBJID"].unique()[0]]
print(df["USUBJID"].unique())
print(artix.dosimetry_parsing(df))

In [ ]:
import pandas
import artix

df = pandas.read_csv(r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\toxicity_data\20241021_AE_TOX_GEN_LTSI.csv", sep=";", encoding='cp1252')
df = df[df["USUBJID"] == df["USUBJID"].unique()[0]]
print(artix.aetox_parsing(df))

# print(df["USUBJID"].unique())
# print(len(df[df["USUBJID"].astype(int) == 5]))

# create ARTIX

In [1]:
import glob, os, tqdm, random
import artix

path = r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data"
patients = []
for p in tqdm.tqdm(glob.glob(os.path.join(path, "*"))[:3], ncols=50):
    patients.append(artix.load_patient(
        path=p,
        id_map=r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\ARTIX_ID_CORRELATION.xlsx",
        PATIENT_DESCRIPTION_csv=r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\toxicity_data\20241021_PATIENT_DESCRIPTION_LTSI.csv",
        EFFICACY_csv=r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\toxicity_data\20241021_EFFICACY_LTSI.csv",
        SALIVATION_DATA_csv=r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\toxicity_data\20241021_SALIVATION_DATA_LTSI.csv",
        TREATMENT_csv=r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\toxicity_data\20241021_TREATMENT_LTSI.csv",
        DOSIMETRY_csv=r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\toxicity_data\20241021_DOSIMETRY_LTSI.csv",
        MDA_csv=r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\toxicity_data\20241021_MDA_LTSI.csv",
        AETOXGEN_csv=r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\toxicity_data\20241021_AE_TOX_GEN_LTSI.csv",
        log="./log",
        ))

print(len(patients))

  0%|                       | 0/3 [00:00<?, ?it/s]

total data loaded from C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data\001
<class 'dicom_class.RTDOSE'>:	 11
<class 'dicom_class.CBCT'>:	 4
<class 'dicom_class.RTSTRUCT'>:	 6
<class 'dicom_class.CT'>:	 6


 33%|█████          | 1/3 [00:49<01:39, 49.70s/it]

total data loaded from C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data\002
<class 'dicom_class.RTDOSE'>:	 9
<class 'dicom_class.CBCT'>:	 3
<class 'dicom_class.RTSTRUCT'>:	 6
<class 'dicom_class.CT'>:	 6


 67%|██████████     | 2/3 [01:43<00:52, 52.22s/it]

total data loaded from C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data\003
<class 'dicom_class.RTDOSE'>:	 7
<class 'dicom_class.CBCT'>:	 6
<class 'dicom_class.RTSTRUCT'>:	 6
<class 'dicom_class.CT'>:	 6


100%|███████████████| 3/3 [02:24<00:00, 48.10s/it]

3


In [2]:
import pickle

with open(r"C:\Users\bilel.guetarni\Desktop\tmp\artix - Copie.pkl", "wb") as f:
    pickle.dump(patients, f)

## check segmentation

In [ ]:
import dicom_utils
import numpy as np

OAR_to_seg = {7: "parotid_gland_left", 8: "parotid_gland_right", 9: "submandibular_gland_right", 10: "submandibular_gland_left"}
threshold = 0.1

for p in patients:
    print(p.id)
    
    for ct in p.ct:
        print(ct.path)
        if ct.rtstruct is None:
            continue

        # TotalSegmentator must be applied before reloading nii
        out = ct.apply_totalsegmentator()

        ct.load_nii()

        for structure_ID, structure_name in OAR_to_seg.items():
            print(structure_name)
            struct_mask = (out == structure_ID).astype("uint8")

            d = 0
            best_oar = None
            for oar in ct.rtstruct.get_all_OARs():
                voxel_coords = ct.rtstruct.get_contours(oar)
                if voxel_coords:
                    original_mask = dicom_utils.fill_vol_ctrs(struct_mask.shape, voxel_coords)

                    dice = 2 * (np.logical_and(struct_mask, original_mask).sum()) / (struct_mask.sum() + original_mask.sum())
                    if dice > d:
                        d = dice
                        best_oar = oar
            print(f"{d:.2f} {best_oar}")

In [ ]:
import pydicom

for p in patients:
    for ct in p.ct:
        if ct.rtstruct is None:
            continue

        dcm = pydicom.dcmread(ct.rtstruct.get_dcm_path())
        for roi_contour in dcm.ROIContourSequence:
            if not hasattr(roi_contour, "ContourSequence"):
                print(f"[NO]\t {ct.rtstruct.path}")

# plots

In [4]:
def is_float_try(str):
    try:
        float(str)
        return True
    except ValueError:
        return False

In [3]:
import pickle
import artix

# with open(r"C:\Users\bilel.guetarni\Desktop\tmp\artix.pkl", "rb") as f:
with open(r"C:\Users\bilel.guetarni\Desktop\tmp\artix - Copie.pkl", "rb") as f:
    patients = pickle.load(f)

print(len(patients))

3


In [7]:
patients[0].clinical

{'STUDYID': 'ARTIX',
 'USUBJID': 98,
 'CENTERID': 'Rennes',
 'ARMCD': 'Replanification Arm',
 'ACTARMCD': 'Replanification Arm',
 'INCDT': '12/06/2017',
 'RANDT': '20/06/2017',
 'SEX': 'Male',
 'AGE': 49,
 'BMI': 24.11,
 'ECOG': 'Completely ambulatory',
 'FUM': 'Active smoker',
 'PA': '36',
 'OH': 'Weaned',
 'DIAB': 'No diabetes',
 'SKIN': 'No',
 'TTT': 'No',
 'ATCD': 'No',
 'P16': 'No',
 'STAG': 'Stage III',
 'ST_STAG': 'Stage III',
 'ST_IMRT': 'No Tomotherapy',
 'ST_HPV': 'Negative',
 'ST_CHEMO': 'CDDP',
 'HISTO': 'epidermoid carcinoma poorly-differentiated',
 'LAT': 'Bilateral',
 'DIAM': '36',
 'NSTAG': 'N1',
 'GG': 'Yes, homolateral',
 'GG_SIZ': '9',
 'GG_NB': '1',
 'GG_FIX': 'K',
 'LOC': 'Several regions',
 'ITT': 'Yes',
 'MITT': 'Yes',
 'PP': 'Yes',
 'REPCPLT': 'Yes',
 'REPDTC': '03/10/2017',
 'BOREP1': 'Complete response',
 'ORR12': 'Complete response',
 'ORR24': 'Complete response',
 'DCD': 'No',
 'DCDDT': nan,
 'DCDR': nan,
 'PROG': 'No',
 'PROGDT': nan,
 'PROGLOC': nan,
 'PRO

In [ ]:
import re
import pandas

# SSF
df = []
for p in patients:
    df_p = {"id": p.id}
    for k, v in p.clinical.items():
        if not "salivation" in k:
            continue

        if not (isinstance(v, (int, float)) or is_float_try(v)) or pandas.isna(v):
            continue

        if "Inclusion".lower() in k.lower():
            df_p.update({"ssf_0": float(v)})
            continue

        M = re.findall("(M\d+)", k)
        if len(M) > 0:
            M = int(re.findall("\d+", M[0])[0])
            df_p.update({f"ssf_{M}": float(v)})

        df_p.update(p.clinical)
    
    df.append(df_p)
        
df = pandas.DataFrame(df)
df

In [ ]:
import plotly.graph_objects as go

dff = df[df["ssf_0"] >= 500]

k = "Standard Arm"
print(f"number of patients with xerostomia baseline in {k}: ", len(df[(df["ssf_0"] < 500) & (df["arm"] == k)]))
k = "Replanification Arm"
print(f"number of patients with xerostomia baseline in {k}: ", len(df[(df["ssf_0"] < 500) & (df["arm"] == k)]))

st6 = len(dff[(dff["ssf_6"] < 500) & (dff["arm"] == "Standard Arm")])
rp6 = len(dff[(dff["ssf_6"] < 500) & (dff["arm"] == "Replanification Arm")])

st12 = len(dff[(dff["ssf_6"] >= 500) & (dff["ssf_12"] < 500) & (dff["arm"] == "Standard Arm")])
rp12 = len(dff[(dff["ssf_6"] >= 500) & (dff["ssf_12"] < 500) & (dff["arm"] == "Replanification Arm")])

st18 = len(dff[(dff["ssf_6"] >= 500) & (dff["ssf_12"] >= 500) & (dff["ssf_18"] < 500) & (dff["arm"] == "Standard Arm")])
rp18 = len(dff[(dff["ssf_6"] >= 500) & (dff["ssf_12"] >= 500) & (dff["ssf_18"] < 500) & (dff["arm"] == "Replanification Arm")])

st24 = len(dff[(dff["ssf_6"] >= 500) & (dff["ssf_12"] >= 500) & (dff["ssf_18"] >= 500) & (dff["ssf_24"] < 500) & (dff["arm"] == "Standard Arm")])
rp24 = len(dff[(dff["ssf_6"] >= 500) & (dff["ssf_12"] >= 500) & (dff["ssf_18"] >= 500) & (dff["ssf_24"] < 500) & (dff["arm"] == "Replanification Arm")])


fig = go.Figure(data=[
    go.Bar(name="Standard Arm", x=[6, 12, 18, 24], y=[st6, st12, st18, st24]),
    go.Bar(name="Replanification Arm", x=[6, 12, 18, 24], y=[rp6, rp12, rp18, rp24])
])
# Change the bar mode
fig.update_layout(title="number of patients developing xerostomia throughout", barmode='group')
fig.show()



st6 = len(dff[(dff["ssf_6"] >= 500) & (dff["arm"] == "Standard Arm")])
rp6 = len(dff[(dff["ssf_6"] >= 500) & (dff["arm"] == "Replanification Arm")])

st12 = len(dff[(dff["ssf_6"] >= 500) & (dff["ssf_12"] >= 500) & (dff["arm"] == "Standard Arm")])
rp12 = len(dff[(dff["ssf_6"] >= 500) & (dff["ssf_12"] >= 500) & (dff["arm"] == "Replanification Arm")])

st18 = len(dff[(dff["ssf_6"] >= 500) & (dff["ssf_12"] >= 500) & (dff["ssf_18"] >= 500) & (dff["arm"] == "Standard Arm")])
rp18 = len(dff[(dff["ssf_6"] >= 500) & (dff["ssf_12"] >= 500) & (dff["ssf_18"] >= 500) & (dff["arm"] == "Replanification Arm")])

st24 = len(dff[(dff["ssf_6"] >= 500) & (dff["ssf_12"] >= 500) & (dff["ssf_18"] >= 500) & (dff["ssf_24"] >= 500) & (dff["arm"] == "Standard Arm")])
rp24 = len(dff[(dff["ssf_6"] >= 500) & (dff["ssf_12"] >= 500) & (dff["ssf_18"] >= 500) & (dff["ssf_24"] >= 500) & (dff["arm"] == "Replanification Arm")])


fig = go.Figure(data=[
    go.Bar(name="Standard Arm", x=[6, 12, 18, 24], y=[st6, st12, st18, st24]),
    go.Bar(name="Replanification Arm", x=[6, 12, 18, 24], y=[rp6, rp12, rp18, rp24])
])
# Change the bar mode
fig.update_layout(title="number of patients without xerostomia throughout", barmode='group')
fig.show()





st6 = len(dff[(dff["ssf_6"] >= 500) & (dff["arm"] == "Standard Arm")])
rp6 = len(dff[(dff["ssf_6"] >= 500) & (dff["arm"] == "Replanification Arm")])

st12 = len(dff[(dff["ssf_12"] >= 500) & (dff["arm"] == "Standard Arm")])
rp12 = len(dff[(dff["ssf_12"] >= 500) & (dff["arm"] == "Replanification Arm")])

st18 = len(dff[(dff["ssf_18"] >= 500) & (dff["arm"] == "Standard Arm")])
rp18 = len(dff[(dff["ssf_18"] >= 500) & (dff["arm"] == "Replanification Arm")])

st24 = len(dff[(dff["ssf_24"] >= 500) & (dff["arm"] == "Standard Arm")])
rp24 = len(dff[(dff["ssf_24"] >= 500) & (dff["arm"] == "Replanification Arm")])


fig = go.Figure(data=[
    go.Bar(name="Standard Arm", x=[6, 12, 18, 24], y=[st6, st12, st18, st24]),
    go.Bar(name="Replanification Arm", x=[6, 12, 18, 24], y=[rp6, rp12, rp18, rp24])
])
# Change the bar mode
fig.update_layout(title="number of patients with SSF>500 at any point", barmode='group')
fig.show()

In [ ]:
import plotly.graph_objects as go

no_xbase_6 = len(df[(df["ssf_0"] >= 500) & (df["ssf_6"] < 500)])
xbase_6 = len(df[(df["ssf_0"] < 500) & (df["ssf_6"] < 500)])

no_xbase_12 = len(df[(df["ssf_0"] >= 500) & (df["ssf_12"] < 500)])
xbase_12 = len(df[(df["ssf_0"] < 500) & (df["ssf_12"] < 500)])

no_xbase_18 = len(df[(df["ssf_0"] >= 500) & (df["ssf_18"] < 500)])
xbase_18 = len(df[(df["ssf_0"] < 500) & (df["ssf_18"] < 500)])

no_xbase_24 = len(df[(df["ssf_0"] >= 500) & (df["ssf_24"] < 500)])
xbase_24 = len(df[(df["ssf_0"] < 500) & (df["ssf_24"] < 500)])


fig = go.Figure(data=[
    go.Bar(name="no xero. baseline", x=[6, 12, 18, 24], y=[no_xbase_6, no_xbase_12, no_xbase_18, no_xbase_24]),
    go.Bar(name="xero. baseline", x=[6, 12, 18, 24], y=[xbase_6, xbase_12, xbase_18, xbase_24])
])
# Change the bar mode
fig.update_layout(title="number of patients developing xerostomia throughout", barmode='group')
fig.show()





no_xbase_6 = len(df[(df["ssf_0"] >= 500) & (df["ssf_6"] < 500)])
xbase_6 = len(df[(df["ssf_0"] < 500) & (df["ssf_6"] < 500)])

no_xbase_12 = len(df[(df["ssf_0"] >= 500) & (df["ssf_6"] >= 500) & (df["ssf_12"] < 500)])
xbase_12 = len(df[(df["ssf_0"] < 500) & (df["ssf_6"] >= 500) & (df["ssf_12"] < 500)])

no_xbase_18 = len(df[(df["ssf_0"] >= 500) & (df["ssf_6"] >= 500) & (df["ssf_12"] >= 500) & (df["ssf_18"] < 500)])
xbase_18 = len(df[(df["ssf_0"] < 500) & (df["ssf_6"] >= 500) & (df["ssf_12"] >= 500) & (df["ssf_18"] < 500)])

no_xbase_24 = len(df[(df["ssf_0"] >= 500) & (df["ssf_6"] >= 500) & (df["ssf_12"] >= 500) & (df["ssf_18"] >= 500) & (df["ssf_24"] < 500)])
xbase_24 = len(df[(df["ssf_0"] < 500) & (df["ssf_6"] >= 500) & (df["ssf_12"] >= 500) & (df["ssf_18"] >= 500) & (df["ssf_24"] < 500)])


fig = go.Figure(data=[
    go.Bar(name="no xero. baseline", x=[6, 12, 18, 24], y=[no_xbase_6, no_xbase_12, no_xbase_18, no_xbase_24]),
    go.Bar(name="xero. baseline", x=[6, 12, 18, 24], y=[xbase_6, xbase_12, xbase_18, xbase_24])
])
# Change the bar mode
fig.update_layout(title="number of patients developing xerostomia throughout with filtering", barmode='group')
fig.show()

In [ ]:
print("total")
print(len(df))

print("SSF baseline unknown")
print(len(df[pandas.isna(df["ssf_0"])]))

print("No xerostomia baseline")
print(len(df[df["ssf_0"] >= 500]))

print("xerostomia baseline")
print(len(df[df["ssf_0"] < 500]))

In [ ]:
for p in patients:
    print(p)
    for i in p.ct:
        print(i.get_acquisition_date())

    p.sort_imaging()
    print()
    print()
    for i in p.ct:
        print(i.get_acquisition_date())

    print()

In [ ]:
import re
import pandas

# SSF
df = []
for p in patients:
    for k, v in p.clinical.items():
        if not "salivation" in k:
            continue

        if not (isinstance(v, (int, float)) or is_float_try(v)):
            continue

        if "Inclusion".lower() in k.lower():
            df.append({"M": 0, "SSF": float(v), "id": p.id, "arm": p.clinical["arm"]})
            continue

        M = re.findall("(M\d+)", k)
        # print(k, M)
        if len(M) > 0:
            M = int(re.findall("\d+", M[0])[0])
            df.append({"M": M, "SSF": float(v), "id": p.id, "arm": p.clinical["arm"]})
        
df = pandas.DataFrame(df)
print(df)

In [ ]:
import plotly.express as px

fig = px.box(df, x="M", y="SSF", color="arm", title="Simulated salivarry flow in mg/min (M=months)")
fig.add_hline(y=500, line_dash="dot",
              annotation_text="xerostomia threshold", 
              annotation_position="bottom left")
fig.show()

In [ ]:
import pandas

map_names = {
    'PAROH_DOSE': ("ipsilateral", "parotid"),
    'PAROC_DOSE': ("contralateral", "parotid"),
    'SMAXH_DOSE': ("ipsilateral", "submandibular"),
    'SMAXC_DOSE': ("contralateral", "submandibular"),
    'MOUTH_DOSE': ("both", "mouth"),
 }

df = []
for p in patients:
    for k, v in p.clinical.items():
        if (not k in map_names.keys()) or (isinstance(v, str) and not is_float_try(v)):
            continue

        side, oar = map_names[k]
        df.append({"id": p.id, "arm": p.clinical["arm"], "side": side, "oar": oar, "dose": float(v)})
        
df = pandas.DataFrame(df)
print(df)

In [ ]:
import plotly.express as px
import numpy as np

fig = px.box(df, x="arm", y="dose", color="side", facet_col="oar")
fig.show()